# Reconstrucción v2
Nueva ejecución de la receta auditada; no se reutilizan salidas ni run_id de v1. Misma entrada congelada y folds. La selección de modelos incluirá seis candidatos y el criterio del negocio desde el diseño. 2014J es un benchmark previamente observado.

# 02 · Features y preparación sin fuga

Dos derivadas motivadas por EDA: log1p de clics actuales e intensidad por día activo. Se implementan en src/pipeline.py, que también centraliza imputación, codificación y escalado. Se analiza solo desarrollo día 56.

Esta etapa verifica las transformaciones; no entrena clasificadores. La evidencia de aporte predictivo por ablación se obtendrá **después de las dos líneas base**, sin abrir hold-out.

In [ ]:
from pathlib import Path
import sys,json,hashlib,subprocess
BASE=Path.cwd()
if BASE.name=='notebooks': BASE=BASE.parent
if not (BASE/'src').is_dir(): BASE=BASE/'pset-03'
sys.path.insert(0,str(BASE))
from IPython.display import display,Markdown
from src.features_evidencia import validar
from src.pipeline import construir_pipeline,construir_preparacion,FeaturesAlCorte
from src.tracking import configurar
from src.common import sha256
summary,folds,descriptivos,contrato=validar()
display(__import__('pandas').DataFrame(contrato['features']))
print('Snapshot:',summary['snapshot_id'])
print('Split:',summary['split_id'])

## 1. Evidencia, fórmulas y disponibilidad

La cola de clics (mediana 53, máximo 7028) motiva log1p, definido también en cero. Es monótona y no agrega información nueva a los clics: puede cambiar la representación útil para modelos lineales/redes, sin prometer mejora en árboles.

La intensidad combina clics y días activos observados en la misma ventana de 14 días. Si no hay días activos, se conserva NaN; el Pipeline imputa y agrega indicador con estadísticas de entrenamiento. No se confunde intensidad desconocida con cero. Las medianas por clase son evidencia descriptiva de desarrollo, no prueba de aporte ni resultados de un clasificador.

Ambas usan únicamente columnas agregadas por Spark al corte. La protección temporal hereda las pruebas de perturbación del PSet 2; aquí se comprueba la invariancia frente a columnas futuras ajenas al contrato y se rechaza un último evento posterior al corte. Esto no certifica la fecha institucional de disponibilidad de todos los atributos.

In [ ]:
display(descriptivos)
print('Casos sin días activos actuales:',summary['zero_current_active_days'])
assert summary['zero_current_active_days']==summary['derived_missing']
print('No se eliminaron observaciones ni se reemplazó indefinido por cero.')

## 2. Preparación aprendida exclusivamente en cada fold

FeaturesAlCorte selecciona por nombres y descarta etiquetas, IDs, folds y auditoría como entradas del modelo. Convierte pd.NA a np.nan. Numéricas: mediana e indicadores de faltantes, luego StandardScaler. Una columna completamente ausente en fit se conserva con fallback cero e indicador. Categóricas: nivel fijo __MISSING__, one-hot y categorías desconocidas ignoradas. El nivel ausente no tiene orden numérico.

Las flags is_imd_missing y has_recorded_activity son explícitas; recencia ausente no se convierte en recencia cero antes del Pipeline. No se recortan extremos. Un pipeline estático exige solo entradas estáticas y nunca depende de actividad; las dos variantes longitudinales comparten bases y difieren solo en las derivadas.

Se ajustan preparadores en los cinco train-folds de cada variante. La validación transforma sin fit. Se contrastan medianas/escalado con los datos del train-fold y se exige que transform no cambie el estado aprendido.

In [ ]:
display(folds)
assert len(folds)==15
assert folds[['salida_finita','medianas_solo_fit','escalado_solo_fit','transform_sin_reajuste','etiquetas_ids_futuro_ignorados']].all().all()
assert folds.alumnos_compartidos.eq(0).all()
print('15 ajustes de preparación verificados; cero clasificadores entrenados.')

## 3. Comparación futura y límites

La siguiente etapa ejecutará primero DummyClassifier y la política reactiva. Después comparará candidatos y ablación con exactamente los folds guardados: estático, longitudinal sin derivadas y con derivadas. El aporte individual requiere además activar cada derivada por separado y reportar la diferencia frente a la dispersión entre folds; no se atribuye aquí una mejora a ninguna.

La selección se hará sin consultar prueba. La codificación densa se mantendrá igual para todas las familias y se medirá memoria/tiempo. No se ha cambiado el protocolo congelado de particiones; este contrato versiona la preparación posterior a EDA.

In [ ]:
import mlflow
experiment=configurar()
source_nb=__import__('nbformat').read(BASE/'notebooks/02_features.ipynb',as_version=4)
source_hash=hashlib.sha256(json.dumps([(c.cell_type,c.source) for c in source_nb.cells],ensure_ascii=False).encode()).hexdigest()
commit=subprocess.run(['git','rev-parse','--verify','HEAD'],cwd=BASE.parent,capture_output=True,text=True)
from src.seguimiento import registrar_base
with mlflow.start_run(run_name='features-preparacion-dia56') as run:
    registrar_base('features', 'preparacion_sin_clasificador', require_frozen=False)
    mlflow.set_tags({'fase':'features','familia':'preparacion_sin_clasificador','cuenta_para_presupuesto_minimo':'false',
                     'git_commit':commit.stdout.strip() if commit.returncode==0 else 'uncommitted',
                     'notebook_source_sha256':source_hash,'pipeline_sha256':sha256(BASE/'src/pipeline.py')})
    mlflow.log_params({'snapshot_id':summary['snapshot_id'],'split_id':summary['split_id'],
                      'snapshot_at_utc':summary['snapshot_at_utc'],'cutoff_day':56,'seed':42,
                      'source_table':'PSET2_OULAD.ANALYTICS.OBT_FEATURES',
                      'client_lock_sha256':sha256(BASE/'requirements.lock.txt')})
    mlflow.log_metrics({'development_rows':summary['development_rows'],'preparation_fits':15,'classifier_fits':0})
    summary.update(run_id=run.info.run_id,experiment_id=experiment.experiment_id,notebook_source_sha256=source_hash)
    (BASE/'docs/features/resumen.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+chr(10))
    mlflow.log_artifacts(str(BASE/'docs/features'),artifact_path='features')
    for f in ['src/pipeline.py','src/features_evidencia.py','notebooks/02_features.ipynb']:
        mlflow.log_artifact(str(BASE/f),artifact_path='source')
assert mlflow.MlflowClient().get_run(summary['run_id']).info.status=='FINISHED'
print('Corrida features FINISHED:',summary['run_id'])
print('Ablación pendiente. La versión 2 no ha evaluado prueba; 2014J sí fue observado en v1.')